%md
# Configuração do Ambiente:
Instalação de pacotes e depedências

In [0]:
########################################################
#### LIBS
########################################################
import os
import sys
import time
from src.geracao_dados_simulados import produtor
from src.streaming import stream_bronze
from pyspark.sql import functions as F
from pyspark.sql.functions import col


#Definição dos diretórios
Local onde os dados serão salvos

In [0]:
########################################################
#### DIRETÓRIOS
########################################################
EVENTO = "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/streaming/eventos"
CHECKPOINT_VALIDO = "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/streaming/checkpoint_valido"
CHECKPOINT_QUARENTENA = "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/streaming/checkpoint_quarentena"
DESTINO_VALIDO = "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/streaming/destino_valido"
DESTINO_QUARENTENA = "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/streaming/destino_quarentena"


#Simulação dos dados
Produção dos dados simulados. Serão gerados 3000 observações, ou seja, essa função será executada 10 vezes.

In [0]:
########################################################
#### PRODUTOR
########################################################
arquivos = produtor(
    diretorio = EVENTO,
    num_lotes = 10,
    num_eventos = 20,
    tempo = 3)

print(f"\n{len(arquivos)} eventos simulados.")

# Consumidor dos dados
Função que irá armazenar os dados simulados.

In [0]:
########################################################
#### CONSUMIDOR
########################################################

#OBS: INICIALMENTE CÓDIGO ELABORADO EM ARQUIVO .py, NO ENTANTO O SCRIPT RODAVA INDEFINIDAMENTE, MAS AINDA SIM FUNCIONAVA DE ACORDO COM O ESPERADO
#EM FORMATO .ipynb NÃO APRESENTA ESSE PROBLEMA
query_valido, query_quarentena = stream_bronze(
    spark=spark,
    evento = EVENTO,
    checkpoint_valido = CHECKPOINT_VALIDO,
    checkpoint_quarentena = CHECKPOINT_QUARENTENA, 
    destino_valido = DESTINO_VALIDO,
    destino_quarentena = DESTINO_QUARENTENA)
    
query_valido.awaitTermination()
query_quarentena.awaitTermination()
print("Streaming concluído.")

# Validação
Query simples para avaliar se o streaming foi executado corretamente.

In [0]:
########################################################
#### LEITURA DOS DADOS EM STREAMING
########################################################

#
dados_stream_valido = spark.read.format("delta").load(DESTINO_VALIDO)
dados_stream_quarentena = spark.read.format("delta").load(DESTINO_QUARENTENA)

#
print(f"Número de eventos válidos e em quarentena, respectivamente: {dados_stream_valido.count()} e {dados_stream_quarentena.count()}. \nTotalizando {dados_stream_valido.count() + dados_stream_quarentena.count()}")



In [0]:
########################################################
#### DADOS VALIDOS AGREGADOS POR UF
########################################################
display(
    dados_stream_valido.groupBy("sigla_uf")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("media_portugues"), 1).alias("media_saeb_pt"),
        F.round(F.avg("taxa_alfabetizacao"), 1).alias("taxa_media"))
        .orderBy(F.desc("media_saeb_pt"))
)

In [0]:
########################################################
#### DADOS VALIDOS AGREGADOS POR UF
########################################################
display(
    dados_stream_quarentena.groupBy("sigla_uf")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("media_portugues"), 1).alias("media_saeb_pt"),
        F.round(F.avg("taxa_alfabetizacao"), 1).alias("taxa_media"))
        .orderBy(F.desc("media_saeb_pt"))
)